In [1]:
# notebooks/02_change_point_model.ipynb

# ============================================
# Cell 1: Import Libraries
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import stats
from datetime import timedelta

warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

# ============================================
# Cell 2: Load the Data
# ============================================
print("\n" + "="*70)
print("LOADING DATA")
print("="*70)

# Load the data
df = pd.read_csv('data/BrentOilPrices.csv')
print(f"✓ Data loaded: {df.shape[0]} rows")

# Parse dates (handles both formats)
def parse_dates(date_str):
    try:
        return pd.to_datetime(date_str, format='%d-%b-%y')
    except:
        try:
            return pd.to_datetime(date_str, format='%b %d, %Y')
        except:
            return pd.NaT

df['Date'] = df['Date'].apply(parse_dates)
df = df.dropna(subset=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f"✓ Date range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"✓ Total observations: {len(df)}")

# ============================================
# Cell 3: Prepare Data for Analysis
# ============================================
print("\n" + "="*70)
print("PREPARING DATA")
print("="*70)

# Extract price series
prices = df['Price'].values
dates = df['Date'].values
n_obs = len(prices)

# Calculate log returns
log_prices = np.log(prices)
log_returns = np.diff(log_prices)

print(f"✓ Price range: ${prices.min():.2f} to ${prices.max():.2f}")
print(f"✓ Mean price: ${prices.mean():.2f}")
print(f"✓ Price std dev: ${prices.std():.2f}")

print(f"\nLog Returns Statistics:")
print(f"  Mean: {log_returns.mean():.6f}")
print(f"  Std: {log_returns.std():.6f}")
print(f"  Min: {log_returns.min():.6f}")
print(f"  Max: {log_returns.max():.6f}")

# ============================================
# Cell 4: Change Point Detection (Rolling Window Method)
# ============================================
print("\n" + "="*70)
print("CHANGE POINT DETECTION")
print("="*70)

def detect_change_points(data, window_size=30, threshold=2.0):
    """
    Detect change points using rolling window statistics
    Returns indices of detected change points and their scores
    """
    n = len(data)
    scores = np.zeros(n)
    changes = []
    
    for i in range(window_size, n - window_size):
        before = data[i-window_size:i]
        after = data[i:i+window_size]
        
        # Calculate t-statistic
        mean_before = np.mean(before)
        mean_after = np.mean(after)
        std_before = np.std(before)
        std_after = np.std(after)
        
        if std_before > 0 and std_after > 0:
            # Pooled standard deviation
            pooled_std = np.sqrt((std_before**2 + std_after**2) / 2)
            if pooled_std > 0:
                t_stat = abs(mean_after - mean_before) / pooled_std
                scores[i] = t_stat
                
                if t_stat > threshold:
                    changes.append(i)
    
    return changes, scores

# Detect changes with different window sizes
print("Detecting change points with different window sizes...")
window_sizes = [30, 60, 90, 120]
change_points = {}

for window in window_sizes:
    changes, scores = detect_change_points(prices, window_size=window, threshold=2.0)
    change_points[window] = changes
    print(f"  Window {window} days: {len(changes)} change points detected")

# ============================================
# Cell 5: Visualize Detected Change Points
# ============================================
print("\n" + "="*70)
print("VISUALIZING CHANGE POINTS")
print("="*70)

fig, axes = plt.subplots(4, 1, figsize=(15, 14))

for idx, (window, changes) in enumerate(change_points.items()):
    ax = axes[idx]
    
    # Plot price series
    ax.plot(dates, prices, color='darkblue', linewidth=0.8, label='Brent Oil Price')
    
    # Mark change points
    for change_idx in changes:
        if change_idx < len(dates):
            ax.axvline(x=dates[change_idx], color='red', linestyle='--', 
                      alpha=0.5, linewidth=1)
            # Add date label for major changes
            if prices[change_idx] > 50:  # Only label significant price points
                ax.text(dates[change_idx], prices[change_idx] + 3, 
                       dates[change_idx].strftime('%Y-%m'), 
                       rotation=45, fontsize=7, ha='center')
    
    ax.set_title(f'Change Points - Window: {window} days ({len(changes)} changes)', 
                 fontsize=14, fontweight='bold')
    ax.set_ylabel('Price (USD/barrel)')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig('figures/change_points_detected.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Figure saved: figures/change_points_detected.png")

# ============================================
# Cell 6: Use 60-day window for further analysis
# ============================================
print("\n" + "="*70)
print("SELECTING CHANGE POINTS FOR ANALYSIS")
print("="*70)

# Use 60-day window (balanced between sensitivity and specificity)
window_choice = 60
selected_changes = change_points[window_choice]
print(f"Using {window_choice}-day window: {len(selected_changes)} change points")

# Get dates for change points
change_dates = []
for change_idx in selected_changes:
    if change_idx < len(dates):
        change_dates.append(dates[change_idx])

print(f"\nTop 10 Change Point Dates:")
for i, date in enumerate(change_dates[:10]):
    print(f"  {i+1}. {date.strftime('%Y-%m-%d')}")

# ============================================
# Cell 7: Load Events Dataset
# ============================================
print("\n" + "="*70)
print("LOADING EVENTS DATASET")
print("="*70)

# Create events dataset (15 key events)
events_data = [
    ('1990-08-02', 'Gulf War - Kuwait Invasion', 'Geopolitical_Conflict', 'High'),
    ('1991-01-17', 'Operation Desert Storm', 'Geopolitical_Conflict', 'High'),
    ('1997-07-02', 'Asian Financial Crisis', 'Economic_Shock', 'Medium'),
    ('2001-09-11', '9/11 Attacks', 'Geopolitical_Event', 'Medium'),
    ('2003-03-20', 'Iraq War Invasion', 'Geopolitical_Conflict', 'High'),
    ('2008-09-15', 'Global Financial Crisis', 'Economic_Shock', 'Very_High'),
    ('2008-12-17', 'OPEC Production Cuts', 'OPEC_Policy', 'High'),
    ('2011-02-15', 'Arab Spring - Libya', 'Geopolitical_Conflict', 'High'),
    ('2014-06-01', 'Oil Price Crash', 'Market_Event', 'High'),
    ('2016-11-30', 'OPEC+ Agreement', 'OPEC_Policy', 'High'),
    ('2018-05-08', 'US Iran Sanctions', 'Political', 'Medium'),
    ('2020-01-30', 'COVID-19 Pandemic', 'Public_Health', 'Very_High'),
    ('2020-03-06', 'OPEC+ Dispute', 'OPEC_Policy', 'Very_High'),
    ('2022-02-24', 'Russia-Ukraine War', 'Geopolitical_Conflict', 'Very_High')
]

events_df = pd.DataFrame(events_data, 
                        columns=['Date', 'Event_Name', 'Category', 'Expected_Impact'])
events_df['Date'] = pd.to_datetime(events_df['Date'])

# Save events
events_df.to_csv('data/events_dataset.csv', index=False)
print(f"✓ Events dataset created: {len(events_df)} events")

print("\nEvents:")
for idx, row in events_df.iterrows():
    print(f"  {row['Date'].strftime('%Y-%m-%d')}: {row['Event_Name']} ({row['Expected_Impact']})")

# ============================================
# Cell 8: Associate Events with Change Points
# ============================================
print("\n" + "="*70)
print("ASSOCIATING EVENTS WITH CHANGE POINTS")
print("="*70)

def find_closest_change(event_date, change_dates, max_days=90):
    """
    Find the closest change point to an event date
    Returns (closest_date, days_diff, price_before, price_after)
    """
    closest = None
    min_diff = float('inf')
    
    for change_date in change_dates:
        diff = abs((change_date - event_date).days)
        if diff < min_diff:
            min_diff = diff
            closest = change_date
    
    if closest is not None and min_diff <= max_days:
        # Find price before and after
        idx = np.where(dates == closest)[0][0] if closest in dates else None
        if idx is not None and idx >= 30 and idx < len(prices) - 30:
            price_before = prices[idx - 30]
            price_after = prices[idx + 30]
            pct_change = ((price_after / price_before) - 1) * 100
            return closest, min_diff, price_before, price_after, pct_change
    
    return None, None, None, None, None

# Create associations
associations = []
change_dates_list = change_dates

for idx, event in events_df.iterrows():
    event_date = event['Date']
    closest_date, days_diff, price_before, price_after, pct_change = find_closest_change(
        event_date, change_dates_list, max_days=90
    )
    
    if closest_date is not None:
        associations.append({
            'Event': event['Event_Name'],
            'Event_Date': event_date,
            'Category': event['Category'],
            'Expected_Impact': event['Expected_Impact'],
            'Change_Date': closest_date,
            'Days_Difference': days_diff,
            'Price_Before': price_before,
            'Price_After': price_after,
            'Price_Change': price_after - price_before,
            'Percent_Change': pct_change
        })

associations_df = pd.DataFrame(associations)

print(f"✓ Found {len(associations_df)} associations")
print("\nAssociations:")
for idx, row in associations_df.iterrows():
    print(f"\n{row['Event']}")
    print(f"  Event Date: {row['Event_Date'].strftime('%Y-%m-%d')}")
    print(f"  Change Point: {row['Change_Date'].strftime('%Y-%m-%d')}")
    print(f"  Days Apart: {row['Days_Difference']}")
    print(f"  Price Change: ${row['Price_Change']:.2f} ({row['Percent_Change']:.1f}%)")

# ============================================
# Cell 9: Visualize Events vs Change Points
# ============================================
fig, ax = plt.subplots(figsize=(16, 8))

# Plot price
ax.plot(dates, prices, color='darkblue', linewidth=1, label='Brent Oil Price')

# Mark change points
for change_date in change_dates_list[:20]:  # Show first 20
    ax.axvline(x=change_date, color='gray', linestyle='--', alpha=0.3, linewidth=0.8)

# Mark events and their associated changes
for idx, row in associations_df.iterrows():
    event_date = row['Event_Date']
    change_date = row['Change_Date']
    
    # Get price at event
    event_idx = np.argmin(np.abs(dates - event_date))
    if event_idx < len(dates):
        price_at_event = prices[event_idx]
        
        # Mark event
        ax.scatter(event_date, price_at_event, color='red', s=100, zorder=5)
        
        # Add label (truncate if too long)
        label = row['Event'][:20]
        if len(row['Event']) > 20:
            label += '...'
        
        ax.text(event_date, price_at_event + 5, label, 
               rotation=45, fontsize=8, ha='center',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

ax.set_title('Brent Oil Prices with Events and Detected Change Points', 
             fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD/barrel)')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig('figures/events_vs_changes.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Figure saved: figures/events_vs_changes.png")

# ============================================
# Cell 10: Quantify Impact by Event Category
# ============================================
print("\n" + "="*70)
print("IMPACT QUANTIFICATION BY CATEGORY")
print("="*70)

# Group by category
category_impact = associations_df.groupby('Category').agg({
    'Percent_Change': ['mean', 'std', 'count', 'min', 'max']
}).round(2)

print("Impact by Event Category:")
print(category_impact)

# Group by expected impact
impact_impact = associations_df.groupby('Expected_Impact').agg({
    'Percent_Change': ['mean', 'std', 'count']
}).round(2)

print("\nImpact by Expected Impact Level:")
print(impact_impact)

# ============================================
# Cell 11: Top Most Significant Events
# ============================================
print("\n" + "="*70)
print("TOP 10 MOST SIGNIFICANT EVENTS")
print("="*70)

# Sort by absolute percent change
top_events = associations_df.sort_values('Percent_Change', ascending=False).head(10)

for idx, row in top_events.iterrows():
    print(f"\n{row['Event']}")
    print(f"  Date: {row['Event_Date'].strftime('%Y-%m-%d')}")
    print(f"  Change Point: {row['Change_Date'].strftime('%Y-%m-%d')}")
    print(f"  Price Before: ${row['Price_Before']:.2f}")
    print(f"  Price After: ${row['Price_After']:.2f}")
    print(f"  Change: ${row['Price_Change']:.2f} ({row['Percent_Change']:.1f}%)")
    print(f"  Category: {row['Category']}")
    print(f"  Impact Level: {row['Expected_Impact']}")
    print(f"  Days from event: {row['Days_Difference']}")

# ============================================
# Cell 12: Statistical Summary
# ============================================
print("\n" + "="*70)
print("STATISTICAL SUMMARY")
print("="*70)

print("\nPrice Statistics:")
print(f"  Overall Mean: ${prices.mean():.2f}")
print(f"  Overall Std: ${prices.std():.2f}")
print(f"  Min: ${prices.min():.2f}")
print(f"  Max: ${prices.max():.2f}")

print(f"\nChange Point Statistics:")
print(f"  Total Change Points: {len(selected_changes)}")
print(f"  Total Events Associated: {len(associations_df)}")
print(f"  Association Rate: {len(associations_df)/len(events_df)*100:.1f}%")

print(f"\nImpact Statistics (Events with changes):")
print(f"  Average Price Change: ${associations_df['Price_Change'].mean():.2f}")
print(f"  Average Percent Change: {associations_df['Percent_Change'].mean():.1f}%")
print(f"  Max Price Change: ${associations_df['Price_Change'].max():.2f}")
print(f"  Min Price Change: ${associations_df['Price_Change'].min():.2f}")

# ============================================
# Cell 13: Summary Report
# ============================================
print("\n" + "="*70)
print("SUMMARY REPORT: CHANGE POINT ANALYSIS")
print("="*70)

print(f"""
1. DATA OVERVIEW
   - Period: {dates[0].strftime('%Y-%m-%d')} to {dates[-1].strftime('%Y-%m-%d')}
   - Total Observations: {len(prices):,}
   - Price Range: ${prices.min():.2f} to ${prices.max():.2f}
   - Mean Price: ${prices.mean():.2f}

2. CHANGE POINT DETECTION
   - Total Change Points: {len(selected_changes)}
   - Window Size Used: {window_choice} days
   - Threshold: 2.0 standard deviations

3. EVENT ASSOCIATIONS
   - Events Analyzed: {len(events_df)}
   - Events Associated with Changes: {len(associations_df)}
   - Association Rate: {len(associations_df)/len(events_df)*100:.1f}%

4. KEY FINDINGS
   - Most Significant Event: {associations_df.loc[associations_df['Percent_Change'].idxmax(), 'Event']} 
     ({associations_df['Percent_Change'].max():.1f}% change)
   - Highest Impact Category: {associations_df.groupby('Category')['Percent_Change'].mean().idxmax()}

5. RECOMMENDATIONS
   - Monitor geopolitical events for significant price changes
   - OPEC policy decisions consistently show price impact
   - Economic shocks cause rapid price movements
   - Consider implementing early warning system for event detection

6. LIMITATIONS
   - Correlation vs. Causation: Change points indicate statistical breaks, not causality
   - Multiple overlapping events can confuse attribution
   - External factors (exchange rates, GDP) not included
   - Event dates may be approximate (events unfold over time)
""")

print("="*70)
print("✓ CHANGE POINT ANALYSIS COMPLETE!")
print("="*70)

# ============================================
# Cell 14: Save Results
# ============================================
print("\nSaving results...")

# Save associations to CSV
associations_df.to_csv('data/event_associations.csv', index=False)
print("✓ Saved: data/event_associations.csv")

# Save change points to CSV
change_df = pd.DataFrame({
    'Date': change_dates,
    'Index': selected_changes,
    'Price': [prices[i] if i < len(prices) else None for i in selected_changes]
})
change_df.to_csv('data/change_points.csv', index=False)
print("✓ Saved: data/change_points.csv")

print("\n✓ All results saved successfully!")
print("\n" + "="*70)

Libraries imported successfully!

LOADING DATA


FileNotFoundError: [Errno 2] No such file or directory: 'data/BrentOilPrices.csv'